# 01 — Data collection

**CRISP-DM phase: Data Preparation (Bronze layer)**

Goal: identify which products in the Amazon Reviews 2023 (McAuley Lab) `Electronics`
category belong to our 5 target brands' premium ANC over-ear headphone lines, then
pull the matching reviews. Output lands in `data/bronze/` — raw extracted data,
not yet cleaned or joined.

Two-step process, because brand ('store') lives in the **metadata** file, not the
**reviews** file:
1. Load metadata -> filter by brand + specific premium product line + price floor -> get list of `parent_asin`
2. Stream the reviews file -> keep only rows whose `parent_asin` is in that list

In [ ]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd


def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bronze layer:", BRONZE_DIR)

In [ ]:
CATEGORY = "Electronics"

# Brand name -> list of matching strings to look for in the 'store' field.
# Kept as separate strings because Amazon store names are inconsistent
# (e.g. "Sennheiser", "SENNHEISER Electronic Corp", "Bowers & Wilkins", "B&W").
TARGET_BRANDS = {
    "Sennheiser": ["sennheiser"],
    "Bose": ["bose"],
    "Sony": ["sony"],
    "Bang & Olufsen": ["bang & olufsen", "bang and olufsen", "b&o"],
    "Bowers & Wilkins": ["bowers & wilkins", "bowers and wilkins", "b&w"],
}

# IMPORTANT: a generic keyword filter ("headphone", "anc", etc.) is too permissive —
# it catches every headphone a brand has ever sold since 1996 (budget, discontinued,
# sport earbuds, etc.), not just the current premium flagship line. We saw this
# concretely in the EDA (00_dataset_exploration.ipynb): Sony matched 1,154 products
# with a generic filter, most of them clearly non-premium.
#
# Fix: match on each brand's actual flagship premium ANC product LINE name instead
# of a generic word like "headphone". Brand-specific, not a shared keyword list.
PRODUCT_LINES = {
    "Sennheiser": ["momentum"],
    "Bose": ["quietcomfort", "quiet comfort"],
    "Sony": ["wh-1000x", "wh1000x"],
    "Bang & Olufsen": ["beoplay h95", "beoplay hx", "beoplay h9"],
    "Bowers & Wilkins": ["px7", "px8"],
}

# Secondary safety net: even within a matched product line, exclude clear non-headphone
# listings (cases, cables, replacement parts sold under the same product-line name).
EXCLUDE_KEYWORDS = ["case", "cable", "replacement", "cover", "sticker", "skin"]

# Price floor (USD — confirmed from the 'price' field format, e.g. "$299.99").
# Premium ANC headphones from these brands realistically retail well above this
# figure. Missing price (None) is allowed through, to not penalize incomplete
# metadata. See 00_dataset_exploration.ipynb for the price-distribution EDA that
# informed this threshold.
MIN_PRICE = 150

## Step 1 — Load and filter metadata

In [ ]:
meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_meta_{CATEGORY}",
    split="full",
    trust_remote_code=True,
)

print(f"Total {CATEGORY} products in metadata: {len(meta):,}")

In [ ]:
def matches_brand(store, brand_strings):
    if not store:
        return False
    store_lower = store.lower()
    return any(b in store_lower for b in brand_strings)


def matches_product_line(title, line_strings):
    if not title:
        return False
    title_lower = title.lower()
    has_line = any(line in title_lower for line in line_strings)
    has_exclusion = any(k in title_lower for k in EXCLUDE_KEYWORDS)
    return has_line and not has_exclusion


def passes_price_floor(price, min_price):
    if price is None or price == "":
        return True  # missing price -> let it through, don't penalize incomplete metadata
    try:
        return float(price) >= min_price
    except (ValueError, TypeError):
        return True  # unparseable price -> don't let a data-quality issue block a real match


matched_rows = []

for row in meta:
    for brand, brand_strings in TARGET_BRANDS.items():
        if not matches_brand(row.get("store"), brand_strings):
            continue
        if not matches_product_line(row.get("title"), PRODUCT_LINES[brand]):
            continue
        if not passes_price_floor(row.get("price"), MIN_PRICE):
            continue
        matched_rows.append({
            "parent_asin": row["parent_asin"],
            "brand": brand,
            "title": row["title"],
            "average_rating": row.get("average_rating"),
            "rating_number": row.get("rating_number"),
            "price": row.get("price"),
        })
        break

products_df = pd.DataFrame(matched_rows)
print(f"Matched products: {len(products_df)}")
products_df["brand"].value_counts()

In [ ]:
# Quick sanity check before moving on — inspect a few titles per brand to confirm
# the filter is actually catching the right kind of product.
for brand in TARGET_BRANDS:
    sample = products_df[products_df["brand"] == brand]["title"].head(3).tolist()
    print(f"\n{brand}:")
    for t in sample:
        print(" -", t)

**Known finding (documented, not a bug):** Bang & Olufsen matches only ~6 distinct
products under this filter, versus dozens for the other brands. This reflects real
market reality — B&O is a design/luxury niche player with far lower sales volume
than mass-market brands like Sony or Bose — not a flaw in the filter logic.

In [ ]:
products_df.to_csv(BRONZE_DIR / "matched_products.csv", index=False)
print("Saved:", BRONZE_DIR / "matched_products.csv")

## Step 2 — Pull matching reviews

The full Electronics reviews file is very large, so we stream it instead of loading
everything into memory, and keep only rows whose `parent_asin` is in our matched set.

This step also includes resilience against network interruptions (observed
repeatedly with this specific large file): retries with exponential backoff, and
incremental checkpointing so a dropped connection never costs more than
`CHECKPOINT_EVERY` reviews of progress.

In [ ]:
import os
import time
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")  # optional but recommended — raises rate limits and often improves stability

target_asins = set(products_df["parent_asin"])
asin_to_brand = dict(zip(products_df["parent_asin"], products_df["brand"]))

reviews_stream = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_review_{CATEGORY}",
    split="full",
    streaming=True,
    trust_remote_code=True,
    token=HF_TOKEN,
)

MAX_REVIEWS_PER_BRAND = 500
CHECKPOINT_PATH = BRONZE_DIR / "premium_headphone_reviews_partial.csv"

# Resume from a previous partial run, if one exists — avoids losing hours of
# progress if the connection drops again.
matched_reviews = []
brand_counts = {b: 0 for b in TARGET_BRANDS}
if CHECKPOINT_PATH.exists():
    existing = pd.read_csv(CHECKPOINT_PATH)
    matched_reviews = existing.to_dict("records")
    brand_counts.update(existing["brand"].value_counts().to_dict())
    print(f"Resuming from checkpoint: {len(matched_reviews)} reviews already collected")
    print(brand_counts)

CHECKPOINT_EVERY = 100  # save progress every N new matched reviews
MAX_RETRIES = 5

since_last_checkpoint = 0
stream_iter = iter(reviews_stream)

while not all(c >= MAX_REVIEWS_PER_BRAND for c in brand_counts.values()):
    for attempt in range(MAX_RETRIES):
        try:
            review = next(stream_iter)
            break
        except StopIteration:
            review = None
            break
        except Exception as e:
            wait = 2 ** attempt
            print(f"Network error ({e}). Retry {attempt + 1}/{MAX_RETRIES} in {wait}s...")
            time.sleep(wait)
    else:
        print("Max retries reached, stopping and saving what we have so far.")
        break

    if review is None:
        print("Reached end of stream.")
        break

    asin = review.get("parent_asin")
    if asin in target_asins:
        brand = asin_to_brand[asin]
        if brand_counts[brand] < MAX_REVIEWS_PER_BRAND:
            matched_reviews.append({
                "parent_asin": asin,
                "brand": brand,
                "rating": review.get("rating"),
                "title": review.get("title"),
                "text": review.get("text"),
                "timestamp": review.get("timestamp"),
                "verified_purchase": review.get("verified_purchase"),
            })
            brand_counts[brand] += 1
            since_last_checkpoint += 1

            if since_last_checkpoint >= CHECKPOINT_EVERY:
                pd.DataFrame(matched_reviews).to_csv(CHECKPOINT_PATH, index=False)
                since_last_checkpoint = 0
                print(f"Checkpoint saved: {len(matched_reviews)} reviews so far. {brand_counts}")

reviews_df = pd.DataFrame(matched_reviews)
print(f"\nTotal reviews collected: {len(reviews_df)}")
reviews_df["brand"].value_counts()

**Known finding (documented, not a bug):** Bang & Olufsen typically ends around
~312 reviews rather than the 500 cap — the stream reaches its end (`"Reached end
of stream"`) before hitting the cap, because only ~6 matching products exist. This
is a real data-availability constraint, not a collection error. Downstream analysis
should compare brands using **proportions, not raw counts**, to avoid this uneven
sample size skewing conclusions.

In [ ]:
reviews_df.to_csv(BRONZE_DIR / "premium_headphone_reviews.csv", index=False)
print("Saved:", BRONZE_DIR / "premium_headphone_reviews.csv")
reviews_df.sample(5)

## Next notebook

With `data/bronze/matched_products.csv` and `data/bronze/premium_headphone_reviews.csv`
in place, the next step is `02_ground_truth.ipynb`:

1. **Silver layer:** merge both bronze files on `parent_asin` into a single
   `data/silver/reviews_enriched.csv` (review text + brand + price + product rating —
   everything needed for analysis in one table).
2. Sample and manually label a subset of `reviews_enriched.csv` (sentiment + aspect
   scores) to build the ground-truth accuracy benchmark (kept separately in
   `data/ground_truth/`, since it's an evaluation artifact, not a pipeline output).